In [2]:
import pandas as pd
import pandera.pandas as pa

df = pd.read_csv("../data/raw/olist_orders_dataset.csv")

schema = pa.DataFrameSchema({
    "order_id": pa.Column(
        str,
        unique=True,
        nullable=False
    )
})

try:

    schema.validate(df)

    print("TESTE DE UNICIDADE APROVADO")
    print("A coluna 'order_id' possui valores únicos.")
    print(f"Total de registros: {len(df)}")
    print(f"Total de order_id únicos: {df['order_id'].nunique()}")

except pa.errors.SchemaError as erro:

    print("TESTE DE UNICIDADE REPROVADO")
    print("Existem valores duplicados na coluna 'order_id'.")
    print("\nDetalhes do erro:")
    print(erro)

TESTE DE UNICIDADE APROVADO
A coluna 'order_id' possui valores únicos.
Total de registros: 99441
Total de order_id únicos: 99441


In [4]:
import pandas as pd
import pandera.pandas as pa

df = pd.read_csv("../data/raw/olist_orders_dataset.csv")

colunas_obrigatorias = [
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_estimated_delivery_date"
]

schema = pa.DataFrameSchema({
    coluna: pa.Column(
        str,
        nullable=False
    )
    for coluna in colunas_obrigatorias
})

try:
    schema.validate(df)

    print("Teste de completude aprovado.")
    print(f"Total de registros: {len(df)}")
    print(f"Total de colunas avaliadas: {len(colunas_obrigatorias)}")

    for coluna in colunas_obrigatorias:
        preenchidos = df[coluna].notna().sum()
        ausentes = df[coluna].isna().sum()

        print(f"\n{coluna}")
        print(f"Preenchidos: {preenchidos}")
        print(f"Ausentes: {ausentes}")

except pa.errors.SchemaError as erro:

    print("Teste de completude reprovado.")
    print(f"Total de registros: {len(df)}")

    for coluna in colunas_obrigatorias:
        preenchidos = df[coluna].notna().sum()
        ausentes = df[coluna].isna().sum()

        print(f"\n{coluna}")
        print(f"Preenchidos: {preenchidos}")
        print(f"Ausentes: {ausentes}")

    print("\nDetalhes:")
    print(erro)

Teste de completude aprovado.
Total de registros: 99441
Total de colunas avaliadas: 5

order_id
Preenchidos: 99441
Ausentes: 0

customer_id
Preenchidos: 99441
Ausentes: 0

order_status
Preenchidos: 99441
Ausentes: 0

order_purchase_timestamp
Preenchidos: 99441
Ausentes: 0

order_estimated_delivery_date
Preenchidos: 99441
Ausentes: 0


In [5]:
import pandas as pd
import pandera.pandas as pa
from pandera import Check

df = pd.read_csv("../data/raw/olist_orders_dataset.csv")

status_validos = [
    "delivered",
    "shipped",
    "canceled",
    "unavailable",
    "invoiced",
    "processing",
    "created",
    "approved"
]

schema = pa.DataFrameSchema({
    "order_status": pa.Column(
        str,
        nullable=False,
        checks=Check.isin(status_validos)
    )
})

try:
    schema.validate(df)

    total_registros = len(df)
    total_validos = df["order_status"].isin(status_validos).sum()
    total_invalidos = total_registros - total_validos

    print("Teste de validade aprovado.")
    print(f"Total de registros: {total_registros}")
    print(f"Registros válidos: {total_validos}")
    print(f"Registros inválidos: {total_invalidos}")

    print("\nDistribuição dos status:")
    print(df["order_status"].value_counts())

except pa.errors.SchemaError as erro:

    total_registros = len(df)
    total_validos = df["order_status"].isin(status_validos).sum()
    total_invalidos = total_registros - total_validos

    print("Teste de validade reprovado.")
    print(f"Total de registros: {total_registros}")
    print(f"Registros válidos: {total_validos}")
    print(f"Registros inválidos: {total_invalidos}")

    print("\nValores inválidos:")
    print(
        df.loc[
            ~df["order_status"].isin(status_validos),
            "order_status"
        ].unique()
    )

    print("\nDetalhes:")
    print(erro)

Teste de validade aprovado.
Total de registros: 99441
Registros válidos: 99441
Registros inválidos: 0

Distribuição dos status:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [6]:
import pandas as pd
import pandera.pandas as pa
from pandera import Check

df = pd.read_csv("../data/raw/olist_orders_dataset.csv")

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for coluna in date_columns:
    df[coluna] = pd.to_datetime(
        df[coluna],
        errors="coerce"
    )

schema = pa.DataFrameSchema(
    checks=[
        Check(
            lambda df:
            df["order_approved_at"].isna()
            | (
                df["order_approved_at"]
                >= df["order_purchase_timestamp"]
            ),
            error="Data de aprovação anterior à data da compra."
        ),

        Check(
            lambda df:
            df["order_delivered_carrier_date"].isna()
            | (
                df["order_delivered_carrier_date"]
                >= df["order_purchase_timestamp"]
            ),
            error="Data de envio anterior à data da compra."
        ),

        Check(
            lambda df:
            df["order_delivered_customer_date"].isna()
            | (
                df["order_delivered_customer_date"]
                >= df["order_purchase_timestamp"]
            ),
            error="Data de entrega anterior à data da compra."
        )
    ]
)

try:
    schema.validate(df)

    print("Teste de consistência aprovado.")
    print(f"Total de registros avaliados: {len(df)}")

except pa.errors.SchemaError as erro:

    print("Teste de consistência reprovado.")
    print(f"Total de registros avaliados: {len(df)}")

    print("\nDetalhes:")
    print(erro)

Teste de consistência reprovado.
Total de registros avaliados: 99441

Detalhes:
DataFrameSchema 'None' failed element-wise validator number 1: <Check <lambda>: Data de envio anterior à data da compra.> failure cases: b9afddbdcfadc9a87b41a83271c3e888, ad133696906f6a78826daa0911b7daec, 74e033208dc13a7b8127eb8e73d09b76, a6b58794fd2ba533359a76c08df576e3, 5792e0b1c8c8a2bf53af468c9a422c88, c3eb293fd154223498b6551a728203e8, b0c2a7d04b165525254254a728c50a4e, 2033a4586b5bec3229ebc1675a8ae092, 08adcddad19d3acf37d1fa01cb9ded1e, dee6298ce7d1fb2645141ef9972157aa, 4e157a36ea9cf89bde6fff57a780b525, d454d6650d375ebc3f9667a4d2fe161c, a5a9f45d795fafe0077043697f62fd4f, dab23a46abb860b6b4816df41eacd610, b4180badf3ac49892e23d6e0276f1e84, 6c5d1e03493316b72304f07834a03325, ede6a833580744e002b0ee65323a3006, 1227f9af3b2ddb0b95060aab8e5b3dc7, 2996234f359b5a3b99263b0f5b841d86, 108c3238d247dead3fd7423cf8990c5b, 0150004d4d8eb63f9948de164da34e34, da383ff6aa9558f3bd909b1d76789381, 9b087af1d905d8987ae126797c9d15b1, f

In [7]:
import pandas as pd
import pandera.pandas as pa

df = pd.read_csv("../data/raw/olist_orders_dataset.csv")

schema = pa.DataFrameSchema({
    "order_id": pa.Column(
        str,
        nullable=False
    ),

    "customer_id": pa.Column(
        str,
        nullable=False
    ),

    "order_status": pa.Column(
        str,
        nullable=False
    )
})

try:
    schema.validate(df)

    print("Validação estrutural de acurácia aprovada.")
    print(f"Total de registros: {len(df)}")
    print(f"Order IDs preenchidos: {df['order_id'].notna().sum()}")
    print(f"Customer IDs preenchidos: {df['customer_id'].notna().sum()}")
    print(f"Status preenchidos: {df['order_status'].notna().sum()}")

    print("\nA acurácia factual não pode ser confirmada")
    print("sem uma fonte externa de referência.")

except pa.errors.SchemaError as erro:

    print("Validação estrutural de acurácia reprovada.")
    print(f"Total de registros: {len(df)}")
    print("\nDetalhes:")
    print(erro)

Validação estrutural de acurácia aprovada.
Total de registros: 99441
Order IDs preenchidos: 99441
Customer IDs preenchidos: 99441
Status preenchidos: 99441

A acurácia factual não pode ser confirmada
sem uma fonte externa de referência.


In [8]:
import pandas as pd
import pandera.pandas as pa
from pandera import Check

df = pd.read_csv("../data/raw/olist_orders_dataset.csv")

df["order_purchase_timestamp"] = pd.to_datetime(
    df["order_purchase_timestamp"],
    errors="coerce"
)

data_minima = pd.Timestamp("2016-01-01")
data_maxima = pd.Timestamp.now()

schema = pa.DataFrameSchema({
    "order_purchase_timestamp": pa.Column(
        "datetime64[ns]",
        nullable=False,
        checks=[
            Check.ge(data_minima),
            Check.le(data_maxima)
        ]
    )
})

try:
    schema.validate(df)

    total_registros = len(df)
    datas_validas = (
        df["order_purchase_timestamp"]
        .between(data_minima, data_maxima)
        .sum()
    )

    datas_invalidas = total_registros - datas_validas

    print("Teste de atualidade/temporalidade aprovado.")
    print(f"Total de registros: {total_registros}")
    print(f"Datas válidas: {datas_validas}")
    print(f"Datas inválidas: {datas_invalidas}")
    print(f"Data mais antiga: {df['order_purchase_timestamp'].min()}")
    print(f"Data mais recente: {df['order_purchase_timestamp'].max()}")

except pa.errors.SchemaError as erro:

    total_registros = len(df)
    datas_validas = (
        df["order_purchase_timestamp"]
        .between(data_minima, data_maxima)
        .sum()
    )

    datas_invalidas = total_registros - datas_validas

    print("Teste de atualidade/temporalidade reprovado.")
    print(f"Total de registros: {total_registros}")
    print(f"Datas válidas: {datas_validas}")
    print(f"Datas inválidas: {datas_invalidas}")
    print("\nDetalhes:")
    print(erro)

Teste de atualidade/temporalidade aprovado.
Total de registros: 99441
Datas válidas: 99441
Datas inválidas: 0
Data mais antiga: 2016-09-04 21:15:19
Data mais recente: 2018-10-17 17:30:18
